In [2]:
import os, sys; from pathlib import Path
os.chdir(Path.home()/"IAA"); sys.path.insert(0, str(Path.home()/"IAA"))

In [3]:
import copy, numpy as np, pandas as pd
import src.pipeline as P, src.target as T
from src.config import PART1, PART2
from src.glm import BandedTweedieGLM
from src.evaluate import evaluate
cfg = copy.deepcopy(PART2); T.AMOUNTS_NET = True          # corrected target (validate() would set this itself)
claims = P.load_claims(cfg["data_dir"])
def summary(name, line, y, mu, w):
    r = evaluate(pd.Series(y), mu, pd.Series(w), show=False)
    q = pd.qcut(pd.Series(mu / w).rank(method="first"), 10, labels=False).to_numpy()
    return {"line": line, "model": name, "D2": r["D2"], "A/E": r["A/E"],
            "low": y[q == 0].sum() / mu[q == 0].sum(), "top": y[q == 9].sum() / mu[q == 9].sum()}
rows = []
for line in ["motor", "contents"]:
    d = P.build_line(line, cfg, claims); feats = d["feats"]
    tr, va = d["tr"][d["tr"].year == 2023], d["tr"][d["tr"].year == 2024]
    y, w = va.target.to_numpy(), va.weight.to_numpy()
    r_cur, _ = P.predict_rate(line, cfg, tr, va, feats)
    rows.append(summary("current (v5)", line, y, np.asarray(r_cur) * w, w))
    dev = P.developed_table(line, cfg); f = [c for c in feats if c in dev.columns]
    g = BandedTweedieGLM(**PART1["glm"][line]).fit(dev[f], dev.target / dev.weight, dev.weight)
    o_tr, o_va = np.asarray(g.predict_rate(tr[f])), np.asarray(g.predict_rate(va[f]))
    a0 = cfg["glm"][line]["alpha"]
    for a, tag in [(a0, "offset (rule)"), (a0 * 0.3, "offset alpha x0.3"), (a0 * 3, "offset alpha x3")]:
        g2 = BandedTweedieGLM(**dict(cfg["glm"][line], alpha=a)).fit(
            tr[feats], (tr.target / tr.weight).to_numpy() / o_tr, tr.weight.to_numpy() * np.sqrt(o_tr))
        m_tr, m_va = np.asarray(g2.predict_rate(tr[feats])), np.asarray(g2.predict_rate(va[feats]))
        lvl = tr.target.sum() / (o_tr * m_tr * tr.weight.to_numpy()).sum()
        rows.append(summary(tag, line, y, o_va * m_va * lvl * w, w))
    print(f"{line}: developed base on {len(f)} features, residual GLM on {len(feats)}")
R = pd.DataFrame(rows); print(R.round(4).to_string(index=False))

13:55:17 motor: developed training table 979,968 rows
motor: developed base on 42 features, residual GLM on 43
13:56:08 contents: sum_insured x1000 entries divided by 1000: train 57, test 135
13:56:12 contents: developed training table 721,617 rows
contents: developed base on 45 features, residual GLM on 46
    line             model     D2    A/E    low    top
   motor      current (v5) 0.0366 1.0721 0.7980 0.8834
   motor     offset (rule) 0.0368 1.0714 0.7853 0.8909
   motor offset alpha x0.3 0.0369 1.0698 0.7417 0.9024
   motor   offset alpha x3 0.0367 1.0719 0.7845 0.8883
contents      current (v5) 0.0538 0.9403 0.7380 0.9857
contents     offset (rule) 0.0542 0.9415 1.0578 0.9703
contents offset alpha x0.3 0.0513 0.9408 0.9105 0.9625
contents   offset alpha x3 0.0541 0.9417 1.0313 0.9922


In [4]:
from pathlib import Path
d = Path.home()/"IAA"/"DECISIONS.md"; s = d.read_text()
LINE = """- **Result:** motor D² 0.0366 -> 0.0368 (+0.0002), tails alike (0.80 / 0.88 -> 0.79 / 0.89): not adopted.
  Contents 0.0538 -> 0.0542 (+0.0004, 0.0001 short of the rule): not adopted, although the safest decile moves from
  A/E 0.74 to 1.06 (top 0.99 -> 0.97); recorded as an observation, the rule is not changed after the result.
  Sensitivity: alpha x0.3 motor +0.0003, contents -0.0025; x3 motor +0.0001, contents +0.0003. Only one feature is
  emerging-only (43 vs 42), so the developed base already carries the relativities: the structural difference
  from the blind design does not change the score; the decisive difference was the target definition.

"""
key = "### E21:"; assert key in s and "- **Result:** motor D² 0.0366 -> 0.0368" not in s
i = s.find("\n\n", s.find("reported as sensitivity only.", s.find(key))) + 2
d.write_text(s[:i] + LINE + s[i:]); print("inserted")

inserted
